<a href="https://colab.research.google.com/github/Psynotch/firstrep/blob/master/Session6_Full_Critical_Strip_Verification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
%pip -q install python-flint==0.9.0

In [4]:
from flint import acb, acb_series, arb, ctx
from fractions import Fraction
from math import ceil

ctx.prec = 192

print("SESSION 06: FULL CRITICAL STRIP")
print("Precision:", ctx.prec, "bits")

SESSION 06: FULL CRITICAL STRIP
Precision: 192 bits


In [5]:
def fraction_ball(q):
    return arb(f"{q.numerator}/{q.denominator}")


def interval_ball(a, b):
    midpoint = (a + b) / 2
    radius = abs(b - a) / 2

    return arb(
        f"{midpoint.numerator}/{midpoint.denominator}",
        f"{radius.numerator}/{radius.denominator}"
    )


def segment_box(p, q):
    return acb(
        interval_ball(p[0], q[0]),
        interval_ball(p[1], q[1])
    )


# Counterclockwise rectangle: 0 <= sigma <= 1, 10 <= t <= 60
corners = [
    (Fraction(0), Fraction(10)),
    (Fraction(1), Fraction(10)),
    (Fraction(1), Fraction(60)),
    (Fraction(0), Fraction(60)),
    (Fraction(0), Fraction(10))
]

MAX_DEPTH = 24

stats = {
    "certified_pieces": 0,
    "max_depth": 0,
    "unresolved": []
}


def certify_segment(p, q, depth=0):
    stats["max_depth"] = max(stats["max_depth"], depth)

    zeta_box = segment_box(p, q).zeta()
    lower = zeta_box.abs_lower()

    if lower.is_finite() and not lower.contains(0):
        stats["certified_pieces"] += 1
        return True

    if depth >= MAX_DEPTH:
        stats["unresolved"].append((p, q))
        return False

    midpoint = (
        (p[0] + q[0]) / 2,
        (p[1] + q[1]) / 2
    )

    left_ok = certify_segment(p, midpoint, depth + 1)
    right_ok = certify_segment(midpoint, q, depth + 1)

    return left_ok and right_ok


boundary_certified = True

print("FULL CRITICAL-STRIP BOUNDARY CHECK")
print("----------------------------------")

for i in range(4):
    success = certify_segment(corners[i], corners[i + 1])

    print(
        f"Side {i + 1}:",
        "CERTIFIED" if success else "NOT CERTIFIED"
    )

    boundary_certified = boundary_certified and success

print("\nBoundary zero-free:", boundary_certified)
print("Certified subsegments:", stats["certified_pieces"])
print("Maximum subdivision depth:", stats["max_depth"])
print("Unresolved segments:", len(stats["unresolved"]))

if not boundary_certified:
    raise RuntimeError(
        "Boundary certification failed. Stop here and inspect unresolved segments."
    )

FULL CRITICAL-STRIP BOUNDARY CHECK
----------------------------------
Side 1: CERTIFIED
Side 2: CERTIFIED
Side 3: CERTIFIED
Side 4: CERTIFIED

Boundary zero-free: True
Certified subsegments: 2146
Maximum subdivision depth: 13
Unresolved segments: 0


In [6]:
#Rigorous Countor integration
def zeta_log_derivative(z, analytic):
    jet = acb_series([z, 1], prec=2).zeta()
    coeff = jet.coeffs()

    return coeff[1] / coeff[0]


def interpolate_point(p, q, k, n):
    fraction = Fraction(k, n)

    return (
        p[0] + (q[0] - p[0]) * fraction,
        p[1] + (q[1] - p[1]) * fraction
    )


def to_acb(point):
    return acb(
        fraction_ball(point[0]),
        fraction_ball(point[1])
    )


def ceil_fraction(x):
    return (x.numerator + x.denominator - 1) // x.denominator


MAX_PANEL_LENGTH = Fraction(1, 5)
tol = arb("1e-22")

side_integrals = []

print("SPLIT CONTOUR INTEGRATION")
print("-------------------------")

for i in range(4):
    p = corners[i]
    q = corners[i + 1]

    span = max(abs(q[0] - p[0]), abs(q[1] - p[1]))
    n = max(1, ceil_fraction(span / MAX_PANEL_LENGTH))

    edge_total = acb(0)

    print(f"\nSide {i + 1}: {n} panels")

    for k in range(n):
        p1 = interpolate_point(p, q, k, n)
        p2 = interpolate_point(p, q, k + 1, n)

        result = acb.integral(
            zeta_log_derivative,
            to_acb(p1),
            to_acb(p2),
            abs_tol=tol,
            rel_tol=tol,
            deg_limit=80,
            eval_limit=200000,
            depth_limit=5000,
            use_heap=True
        )

        if not result.is_finite():
            raise RuntimeError(
                f"Integration failed on side {i + 1}, panel {k + 1}"
            )

        edge_total += result

    side_integrals.append(edge_total)
    print("Side integral:", edge_total)


total_integral = sum(side_integrals, acb(0))

print("\nTOTAL CONTOUR INTEGRAL")
print("----------------------")
print(total_integral)

SPLIT CONTOUR INTEGRATION
-------------------------

Side 1: 5 panels
Side integral: [-0.232354013292327389218 +/- 1.30e-22] + [-0.021073270727252294451 +/- 4.92e-22]j

Side 2: 250 panels
Side integral: [-1.01430551597067736784 +/- 5.22e-21] + [0.29775333689901723129 +/- 5.64e-21]j

Side 3: 5 panels
Side integral: [1.128233747906377600635 +/- 3.65e-22] + [0.359058889987519552085 +/- 4.44e-22]j

Side 4: 250 panels
Side integral: [0.1184257813566271564 +/- 3.76e-20] + [81.0456700371753397111 +/- 1.83e-20]j

TOTAL CONTOUR INTEGRAL
----------------------
[+/- 2.02e-20] + [81.6814089933346242000 +/- 4.89e-20]j


In [7]:
#Certify full-strip zero count
two_pi_i = 2 * acb.pi() * acb(0, 1)

zero_count_ball = total_integral / two_pi_i
unique_count = zero_count_ball.unique_fmpz()

print("FULL-STRIP ZERO COUNT ENCLOSURE")
print("--------------------------------")
print(zero_count_ball)

if unique_count is None:
    print("\nNOT YET CERTIFIED")
    print("The enclosure does not isolate one integer.")

else:
    certified_full_count = int(unique_count)

    print("\nCERTIFIED FULL-STRIP ZERO COUNT:",
          certified_full_count)

    print("Matches previous count:",
          certified_full_count == 13)

FULL-STRIP ZERO COUNT ENCLOSURE
--------------------------------
[13.00000000000000000000 +/- 3.21e-21] + [+/- 3.21e-21]j

CERTIFIED FULL-STRIP ZERO COUNT: 13
Matches previous count: True


In [8]:
#Recheck 13 critical-line intervals
def hardy_z(t_fraction):
    t = fraction_ball(t_fraction)

    s = acb(arb(1) / 2, t)
    gamma_arg = acb(arb(1) / 4, t / 2)

    theta = (
        gamma_arg.lgamma().imag
        - (t / 2) * arb.pi().log()
    )

    phase = acb(0, theta).exp()
    result = phase * s.zeta()

    return result.real


zero_centers = [
    "14.13472514173469379045725",
    "21.02203963877155499262848",
    "25.01085758014568876321379",
    "30.4248761258595132103119",
    "32.93506158773918969066237",
    "37.58617815882567125721776",
    "40.91871901214749518739813",
    "43.32707328091499951949612",
    "48.00515088116715972794247",
    "49.77383247767230218191678",
    "52.9703214777144606441473",
    "56.44624769706339480436776",
    "59.34704400260235307965365"
]

radius = Fraction(1, 100)

intervals = [
    (
        Fraction(c) - radius,
        Fraction(c) + radius
    )
    for c in zero_centers
]


def certified_sign(value):
    sign = value.sgn()

    if sign == arb(1):
        return "+"

    if sign == arb(-1):
        return "-"

    return "UNKNOWN"


verified = 0

print("HARDY Z SIGN-CHANGE VERIFICATION")
print("--------------------------------")

for i, (left, right) in enumerate(intervals, 1):
    sign_left = certified_sign(hardy_z(left))
    sign_right = certified_sign(hardy_z(right))

    passed = (
        sign_left != "UNKNOWN"
        and sign_right != "UNKNOWN"
        and sign_left != sign_right
    )

    if passed:
        verified += 1

    print(
        f"Zero {i:02d}: "
        f"Z(left)={sign_left}, "
        f"Z(right)={sign_right}, "
        f"PASS={passed}"
    )

print("\nVERIFIED SIGN CHANGES:", verified)

HARDY Z SIGN-CHANGE VERIFICATION
--------------------------------
Zero 01: Z(left)=-, Z(right)=+, PASS=True
Zero 02: Z(left)=+, Z(right)=-, PASS=True
Zero 03: Z(left)=-, Z(right)=+, PASS=True
Zero 04: Z(left)=+, Z(right)=-, PASS=True
Zero 05: Z(left)=-, Z(right)=+, PASS=True
Zero 06: Z(left)=+, Z(right)=-, PASS=True
Zero 07: Z(left)=-, Z(right)=+, PASS=True
Zero 08: Z(left)=+, Z(right)=-, PASS=True
Zero 09: Z(left)=-, Z(right)=+, PASS=True
Zero 10: Z(left)=+, Z(right)=-, PASS=True
Zero 11: Z(left)=-, Z(right)=+, PASS=True
Zero 12: Z(left)=+, Z(right)=-, PASS=True
Zero 13: Z(left)=-, Z(right)=+, PASS=True

VERIFIED SIGN CHANGES: 13


In [9]:
print("SESSION 06 FINAL CHECK")
print("======================")

print("Boundary certified:", boundary_certified)
print("Full-strip count:", unique_count)
print("Hardy sign changes:", verified)

success = (
    boundary_certified
    and unique_count is not None
    and int(unique_count) == 13
    and verified == 13
)

if success:
    print("\nSESSION 06: SUCCESS")
    print("All 13 zeros in the full critical-strip rectangle")
    print("are accounted for by 13 distinct critical-line zeros.")
    print("Each zero is simple.")

else:
    print("\nSESSION 06: NOT YET COMPLETE")
    print("Inspect the failed certification before drawing conclusions.")

SESSION 06 FINAL CHECK
Boundary certified: True
Full-strip count: 13
Hardy sign changes: 13

SESSION 06: SUCCESS
All 13 zeros in the full critical-strip rectangle
are accounted for by 13 distinct critical-line zeros.
Each zero is simple.
